# 🚆 Drishti-Kavach: High-Precision YOLO11-seg Cloud Training on Kaggle

This notebook trains the **Drishti-Kavach (YOLO11-seg)** Railway Hazard & Track Clearance model on **Kaggle's Free Cloud GPUs (NVIDIA T4 x 2 or P100)**.

### ⚡ Why Kaggle is Recommended over Google Colab:
1. **Zero Disconnection Issues:** You can click **"Save Version" -> "Save & Run All (Commit)"**, close your browser/laptop, and Kaggle will run the full 60 epochs in the background (up to 12 continuous hours).
2. **Fast NVMe Dataset Storage:** Datasets added to `/kaggle/input/` are mounted instantly without Google Drive sync bottlenecks or auth dropouts.
3. **Free 30 Hours GPU/week:** Access to Dual NVIDIA T4 (2x16GB) or P100 (16GB) GPUs.
4. **Direct Output Downloads:** The resulting `RailDrishti.pt` and all metrics/plots are saved directly to `/kaggle/working/` for 1-click download.

### ⚙️ Step 1: Verify Kaggle GPU Accelerator & Settings
Make sure that in the right sidebar under **Notebook options / Settings**:
- **Accelerator:** `GPU T4 x 2` (or `GPU P100`)
- **Internet:** `ON` (Required to install ultralytics and download base weights)

In [ ]:
!nvidia-smi
import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu_count = torch.cuda.device_count()
    print(f"Detected {gpu_count} GPU device(s):")
    for i in range(gpu_count):
        print(f"  • GPU {i}: {torch.cuda.get_device_name(i)} (VRAM: {torch.cuda.get_device_properties(i).total_memory / (1024**3):.1f} GB)")

### 📦 Step 2: Install Ultralytics & Core Dependencies

In [ ]:
!pip install -q ultralytics opencv-python-headless shapely pyyaml

### 📂 Step 3: Auto-Detect Dataset & Generate Kaggle YAML Configuration

In [ ]:
import os
import glob
import shutil
import zipfile
from pathlib import Path

# 1. Locate dataset in /kaggle/input
input_root = Path("/kaggle/input")
train_img_dir = None
val_img_dir = None
dataset_root_dir = None

# Check if dataset is already extracted
for p in input_root.rglob("images/train"):
    if p.is_dir():
        train_img_dir = p
        val_img_dir = p.parent / "val"
        dataset_root_dir = p.parent.parent
        break

# If not extracted yet, check for zip archives in /kaggle/input
if train_img_dir is None:
    zip_files = list(input_root.rglob("*.zip"))
    if zip_files:
        target_zip = zip_files[0]
        print(f"[*] Extracting dataset archive {target_zip} to /tmp/dataset...")
        extract_dir = Path("/tmp/dataset")
        with zipfile.ZipFile(str(target_zip), 'r') as z:
            z.extractall(str(extract_dir))
        for p in extract_dir.rglob("images/train"):
            if p.is_dir():
                train_img_dir = p
                val_img_dir = p.parent / "val"
                dataset_root_dir = p.parent.parent
                break

if train_img_dir is None or not train_img_dir.exists():
    raise FileNotFoundError(
        "[!] Could not locate 'images/train' in /kaggle/input. "
        "Please make sure you added the RailDrishti dataset via '+ Add Data' in the right sidebar."
    )

print(f"[✓] Found dataset root at: {dataset_root_dir}")
print(f"    • Train Images: {train_img_dir} ({len(list(train_img_dir.glob('*')))} files)")
print(f"    • Val Images  : {val_img_dir} ({len(list(val_img_dir.glob('*')))} files)")

# 2. Create Dynamic Kaggle Dataset YAML
yaml_path = Path("/kaggle/working/raildrishti_kaggle.yaml")
yaml_content = f"""path: {dataset_root_dir}
train: {train_img_dir}
val: {val_img_dir}
names:
  0: Rail_Track_Bed
  1: Rail_Lines
  2: Person
  3: Car
  4: Truck
  5: Branch
  6: IronRod
  7: Barrel
  8: Boulder
  9: Jerrycan
"""

with open(yaml_path, "w") as f:
    f.write(yaml_content)

print(f"[✓] Dataset YAML written to: {yaml_path}")

### 📊 Step 4: Setup Real-Time Accuracy Telemetry Callback
Tracks Box mAP, Track Mask Segmentation mAP, and automatically preserves the all-time best weights.

In [ ]:
def get_metric_color(val: float) -> str:
    if val >= 90.0: return "\033[1;92m"
    elif val >= 80.0: return "\033[92m"
    elif val >= 65.0: return "\033[93m"
    elif val >= 45.0: return "\033[95m"
    else: return "\033[91m"

class KaggleAccuracyMonitor:
    TARGET_BOX_MAP = 85.0
    TARGET_SEG_MAP = 90.0
    TARGET_OVERALL_MAP = 85.0

    def __init__(self, export_path: Path):
        self.best_map = 0.0
        self.best_epoch = 0
        self.export_path = export_path

    def on_fit_epoch_end(self, trainer):
        epoch = trainer.epoch + 1
        total_epochs = trainer.epochs
        metrics = getattr(trainer, "metrics", {}) or {}
        box_map50 = (metrics.get("metrics/mAP50(B)", 0.0) or 0.0) * 100.0
        seg_map50 = (metrics.get("metrics/mAP50(M)", 0.0) or 0.0) * 100.0
        overall_map50 = (box_map50 + seg_map50) / 2.0 if (box_map50 > 0 and seg_map50 > 0) else (box_map50 or seg_map50 or 0.0)
        
        loss_val = None
        if hasattr(trainer, "tloss") and trainer.tloss is not None:
            try: loss_val = float(trainer.tloss.mean())
            except Exception: pass

        is_new_best = False
        if overall_map50 > self.best_map and overall_map50 > 1.0:
            self.best_map = overall_map50
            self.best_epoch = epoch
            is_new_best = True
            if hasattr(trainer, "best") and os.path.exists(str(trainer.best)):
                try: shutil.copy(str(trainer.best), str(self.export_path))
                except Exception: pass

        c_overall = get_metric_color(overall_map50)
        c_seg = get_metric_color(seg_map50)
        c_box = get_metric_color(box_map50)
        c_best = get_metric_color(self.best_map)
        rst = "\033[0m"
        loss_str = f"{loss_val:.4f}" if loss_val is not None else "N/A"
        best_msg = f" (★ Saved to {self.export_path.name})" if is_new_best else ""

        print(f"\nEpoch [{epoch:02d}/{total_epochs:02d}] -> Overall Accuracy: {c_overall}{overall_map50:.1f}%{rst} (Expected: ≥{self.TARGET_OVERALL_MAP:.0f}%) | Loss: {loss_str}")
        print(f"Metrics: Track Segm mAP: {c_seg}{seg_map50:.1f}%{rst} (Expected: ≥{self.TARGET_SEG_MAP:.0f}%) | Obstacle Box mAP: {c_box}{box_map50:.1f}%{rst} (Expected: ≥{self.TARGET_BOX_MAP:.0f}%)")
        print(f"Best Accuracy: {c_best}{self.best_map:.1f}%{rst} at Epoch {self.best_epoch}{best_msg}\n")

### 🚀 Step 5: Launch Training (High-Precision 1024px)

Configure your hyperparameters below and run the cell.

In [ ]:
import torch
from ultralytics import YOLO

# ── CONFIGURATION ─────────────────────────────────────────────────────────────
EPOCHS = 60                   # Target epochs for convergence
IMGSZ = 1024                  # 1024x1024 High-Precision Railway resolution
BASE_MODEL = "yolo11s-seg.pt" # Pretrained YOLO11 small segmentation model

# Determine optimal batch size & device setup
gpu_count = torch.cuda.device_count() if torch.cuda.is_available() else 0
if gpu_count >= 2:
    DEVICE = [0, 1]           # Utilize both T4 GPUs on Kaggle
    BATCH_SIZE = 16           # 8 images per GPU
    WORKERS = 8
elif gpu_count == 1:
    DEVICE = 0                # Single T4 or P100 GPU
    BATCH_SIZE = 8            # Optimal for 1024px on single 16GB VRAM GPU
    WORKERS = 4
else:
    DEVICE = "cpu"
    BATCH_SIZE = 4
    WORKERS = 2
# ──────────────────────────────────────────────────────────────────────────────

export_model_path = Path("/kaggle/working/RailDrishti.pt")
monitor = KaggleAccuracyMonitor(export_path=export_model_path)

print("=" * 75)
print(" 🚆 STARTING DRISHTI-KAVACH TRAINING ON KAGGLE CLOUD GPU")
print("=" * 75)
print(f" • Base Model  : {BASE_MODEL}")
print(f" • Resolution  : {IMGSZ}x{IMGSZ} px")
print(f" • Device(s)   : {DEVICE} ({gpu_count} GPU detected)")
print(f" • Batch Size  : {BATCH_SIZE}")
print(f" • Epochs      : {EPOCHS}")
print(f" • Output Path : {export_model_path}")
print("=" * 75 + "\n")

# Load pretrained model
model = YOLO(BASE_MODEL)
model.add_callback("on_fit_epoch_end", monitor.on_fit_epoch_end)

# Train with optimized railway hyperparameters
results = model.train(
    data="/kaggle/working/raildrishti_kaggle.yaml",
    epochs=EPOCHS,
    batch=BATCH_SIZE,
    imgsz=IMGSZ,
    device=DEVICE,
    workers=WORKERS,
    project="/kaggle/working/runs",
    name="raildrishti_kaggle",
    exist_ok=True,
    amp=True,
    optimizer="AdamW",
    lr0=0.002,
    cos_lr=True,
    weight_decay=0.0005,
    warmup_epochs=3.0,
    patience=15,
    mosaic=1.0,
    mixup=0.1,
    fliplr=0.5,
    hsv_h=0.015,
    hsv_s=0.6,
    hsv_v=0.4
)

# Ensure best model is saved to /kaggle/working/RailDrishti.pt
best_weights = Path("/kaggle/working/runs/raildrishti_kaggle/weights/best.pt")
if best_weights.exists():
    shutil.copy(str(best_weights), str(export_model_path))
    print(f"\n[✓] Best model exported to: {export_model_path} ({os.path.getsize(export_model_path) / (1024*1024):.1f} MB)")

### 📈 Step 6: Visualize Training Performance & Evaluation Plots

In [ ]:
from IPython.display import Image, display

runs_dir = Path("/kaggle/working/runs/raildrishti_kaggle")

plots_to_show = [
    runs_dir / "results.png",
    runs_dir / "confusion_matrix_normalized.png",
    runs_dir / "BoxPR_curve.png",
    runs_dir / "MaskPR_curve.png",
    runs_dir / "val_batch0_pred.jpg"
]

for plot_path in plots_to_show:
    if plot_path.exists():
        print(f"\n📊 Displaying: {plot_path.name}")
        display(Image(filename=str(plot_path)))
    else:
        print(f"[-] Plot not found: {plot_path.name}")

### 📦 Step 7: Zip Training Results for 1-Click Download
This packages all weights, metrics, curves, and validation predictions into a single zip file in `/kaggle/working/`.

In [ ]:
import zipfile

output_zip = "/kaggle/working/raildrishti_training_results.zip"
source_dir = "/kaggle/working/runs/raildrishti_kaggle"

print(f"[*] Packaging training results to {output_zip}...")
with zipfile.ZipFile(output_zip, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for root, _, files in os.walk(source_dir):
        for file in files:
            full_path = os.path.join(root, file)
            rel_path = os.path.relpath(full_path, source_dir)
            zipf.write(full_path, arcname=rel_path)

print(f"[✓] Done! File size: {os.path.getsize(output_zip) / (1024*1024):.1f} MB")
print("\n🎉 Download your trained model from the right sidebar -> Output section:")
print("  1. RailDrishti.pt (Direct best weights)")
print("  2. raildrishti_training_results.zip (Full metrics, curves & checkpoints)")